# IT23183186 Model B ? FAIRHiveFrames YOLO11n

This notebook runs the repository's dataset validator, training script, and test evaluator. It does not create labels or convert unsuitable annotations. The supported labels are structural comb/frame classes, not disease classes. The dataset is not included in the repository; configure your own prepared copy with valid YOLO bounding boxes and an independent group-aware test split before training. Run cells in order.

## 1. Install dependencies and configure data

Install the model environment once if needed: `python -m pip install -r backend/models/IT23183186_model_b_hive_frame_detector/requirements.txt`. Restart the notebook kernel after installing. A compatible Ultralytics 8.3 release is required; training downloads/loads the official `yolo11n.pt` pretrained checkpoint if it is not already cached.

Set `DATASET_ROOT` below to a **prepared copy** with `images/train`, `images/val`, `images/test` and matching `labels/...` directories. Preserve the original dataset. The test set should be a group-aware holdout that is not used for training or model selection. The class names and their order are fixed by the verified dataset specification.

In [ ]:
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import subprocess
import sys

MODEL_RELATIVE_DIR = Path('backend') / 'models' / 'IT23183186_model_b_hive_frame_detector'

def find_repository_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / MODEL_RELATIVE_DIR).is_dir():
            return candidate
    raise FileNotFoundError(
        f'Could not find the repository above {start}. Start Jupyter inside the project folder.'
    )

REPO_ROOT = find_repository_root()
MODEL_DIR = REPO_ROOT / MODEL_RELATIVE_DIR

# Change this to the root of your prepared dataset. Do not point it at the original archive.
DATASET_ROOT = Path(r'C:\datasets\FAIRHiveFrames_prepared')
DATA_YAML = DATASET_ROOT / 'dataset.yaml'
RUN_ID = datetime.now().strftime('IT23183186_ModelB_%Y%m%d_%H%M%S')
RUNS_DIR = MODEL_DIR / 'runs'
AUDIT_REPORT = MODEL_DIR / 'reports' / f'dataset_audit_{RUN_ID}.json'
RUN_NAME = f'IT23183186_ModelB_YOLO11n_{RUN_ID[-15:]}'
EVALUATION_NAME = f'{RUN_NAME}_test'
PYTHON = sys.executable

CLASS_NAMES = [
    'CappedHoneyCell', 'CappedWorkerBroodCell', 'EmptyCombCell',
    'PollenCell', 'UncappedNectarCell', 'UncappedWorkerLarvaCell', 'BeeHiveFrame',
]

def run(args):
    command = [str(value) for value in args]
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)

print('Repository root:', REPO_ROOT)
print('Model scripts:', MODEL_DIR)
print('Prepared dataset root exists:', DATASET_ROOT.is_dir())
try:
    print('Ultralytics version:', version('ultralytics'))
except PackageNotFoundError:
    print('Ultralytics is not installed in this notebook kernel; install the requirements above.')


## 2. Create or check the dataset YAML

If `dataset.yaml` is absent, the cell creates it inside the prepared dataset folder using the conventional split folders and the fixed class map. This creates configuration only; it does not create or change any annotations. If a YAML already exists, it is preserved and checked. Make sure its paths refer to the actual prepared split folders.

In [ ]:
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f'Prepared dataset folder not found: {DATASET_ROOT}. Set DATASET_ROOT to your local prepared copy.'
    )

if not DATA_YAML.exists():
    import yaml
    required_dirs = [DATASET_ROOT / 'images' / split for split in ('train', 'val', 'test')]
    missing = [str(path) for path in required_dirs if not path.is_dir()]
    if missing:
        raise FileNotFoundError('Expected images/train, images/val, and images/test. Missing: ' + ', '.join(missing))
    config = {
        'path': DATASET_ROOT.resolve().as_posix(),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {index: name for index, name in enumerate(CLASS_NAMES)},
    }
    DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    print('Created dataset config:', DATA_YAML)
else:
    print('Using existing dataset config:', DATA_YAML)

sys.path.insert(0, str(MODEL_DIR))
from model import read_dataset_yaml
config, resolved_root = read_dataset_yaml(DATA_YAML, require_test=True)
print('Resolved dataset root:', resolved_root)
print('Splits:', {name: config[name] for name in ('train', 'val', 'test')})
print('Verified class names:', config['names'])

## 3. Validate images, labels, boxes, duplicates, and split separation

This check must pass before training. It checks image/label matches, readable images, YOLO box format and bounds, class IDs, exact duplicates, near-duplicate candidates, and detectable group overlap. Review the saved JSON report as well. If it fails or flags near-duplicate candidates, resolve the dataset issue without inventing labels, then rerun validation.

In [ ]:
run([
    PYTHON, MODEL_DIR / 'validate_dataset.py',
    '--data', DATA_YAML,
    '--output', AUDIT_REPORT,
])

import json
audit = json.loads(AUDIT_REPORT.read_text(encoding='utf-8'))
if audit.get('passed') is not True:
    raise RuntimeError(f'Dataset audit did not pass. Review {AUDIT_REPORT}')
print('Dataset validation passed. Report:', AUDIT_REPORT)

## 4. Train YOLO11n

Run after the dataset audit passes. Training uses the training split and validation split for early stopping and checkpoint selection. It leaves the test split for the final evaluation cell. The unique run name prevents an earlier run from being overwritten. Default settings are 100 epochs maximum, 640 image size, batch 8, patience 20, and seed 2026; adjust batch size if memory is limited.

In [ ]:
# Refuse to train unless this notebook's saved validation audit passed.
import json
audit = json.loads(AUDIT_REPORT.read_text(encoding='utf-8'))
if audit.get('passed') is not True:
    raise RuntimeError(f'Training blocked: dataset audit did not pass. Review {AUDIT_REPORT}')

run([
    PYTHON, MODEL_DIR / 'train.py',
    '--data', DATA_YAML,
    '--epochs', '100',
    '--imgsz', '640',
    '--batch', '8',
    '--patience', '20',
    '--workers', '4',
    '--seed', '2026',
    '--project', RUNS_DIR,
    '--name', RUN_NAME,
])

BEST_WEIGHTS = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
print('Best checkpoint:', BEST_WEIGHTS)
print('Checkpoint exists:', BEST_WEIGHTS.is_file())

## 5. Evaluate once on the held-out test split

Run only after training finishes and `best.pt` exists. The evaluation script selects `split=test` and writes actual metrics and plots to a separate run directory. These measurements describe detection of the named hive structures; they do not indicate Varroa, AFB, EFB, or any other disease.

In [ ]:
BEST_WEIGHTS = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f'Best checkpoint not found: {BEST_WEIGHTS}. Complete the training cell first.')
run([
    PYTHON, MODEL_DIR / 'evaluate.py',
    '--data', DATA_YAML,
    '--weights', BEST_WEIGHTS,
    '--imgsz', '640',
    '--batch', '8',
    '--project', RUNS_DIR,
    '--name', EVALUATION_NAME,
])
print('Evaluation report:', RUNS_DIR / EVALUATION_NAME / 'test_evaluation.json')

## Outputs

- Dataset audit: `backend/models/IT23183186_model_b_hive_frame_detector/reports/dataset_audit_<run>.json`
- Best checkpoint: `backend/models/IT23183186_model_b_hive_frame_detector/runs/<run>/weights/best.pt`
- Test report and plots: `backend/models/IT23183186_model_b_hive_frame_detector/runs/<test-run>/`

For Model A, artifacts are written under `artifacts/<run>/` and its test metrics are in `artifacts/<run>/evaluation/test_evaluation.json`. Training and test evaluation are intentionally separate notebook cells.